# DesignGym GRPO — Hackathon Submission

**Base → SFT → GRPO** 3-way comparison on the DesignGym RL environment.

### How this notebook works
This Colab does **NOT** use Colab's GPU. Instead, it submits a HuggingFace Job that runs on a Tesla T4 (paid via your HF credits, ~$0.40/run), streams the live logs back to this cell, then downloads and renders the resulting tables and plots inline.

### Configuration (full run, not smoke)
- **400 env states** (training dataset)
- **200 GRPO steps**
- **3 eval seeds × 3 tasks = 9 rollouts per model**
- Uploads adapter + plots + CSVs to `yashvyasop/designgym2-grpo-qwen05-lora`

### Setup before running
1. Add `HF_TOKEN` to **Colab secrets** (left sidebar → key icon → name it `HF_TOKEN`)
2. Just run all cells. No GPU runtime needed — CPU runtime works fine.

## 1. Install HuggingFace CLI

In [ ]:
%%capture
!pip install -U "huggingface_hub[cli]>=0.30" pandas matplotlib

## 2. Login to HuggingFace

In [ ]:
import os
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", "")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN missing. Add it to Colab secrets: left sidebar → key icon → HF_TOKEN"
    )

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
USERNAME = whoami(token=HF_TOKEN)["name"]
print(f"Logged in as: {USERNAME}")

## 3. Submit GRPO Training Job to HuggingFace + Stream Live Logs

This cell:
1. Submits a Tesla T4 job to HF (uses your HF credits, **not** Colab's GPU).
2. Runs the **exact same `run_grpo.sh` that passed the smoke test** — just without `--smoke`, so it picks up the full defaults from `grpo_train.py`:
   - `num_train_states=400`
   - `max_steps=200`
   - `eval_seeds=3`
3. Streams the live job logs into this cell as the job runs.

Expected runtime: **~40 min** on Tesla T4 (~$0.40 of HF credits).

In [ ]:
import subprocess, sys, re

# Identical to the smoke-test command from terminal 5.txt, just without --smoke
JOB_CMD = [
    "hf", "jobs", "run",
    "--flavor", "t4-small",
    "--timeout", "90m",
    "--secrets", f"HF_TOKEN={HF_TOKEN}",
    "pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel",
    "/bin/sh", "-c",
    "apt-get update -y && apt-get install -y git curl && "
    "git clone https://huggingface.co/spaces/yashvyasop/DesignGym /workspace/DesignGym && "
    "cd /workspace/DesignGym && bash training/run_grpo.sh",
]

print("Submitting HF Job…\n" + "=" * 72)

JOB_ID = None
proc = subprocess.Popen(
    JOB_CMD,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
    if JOB_ID is None:
        m = re.search(r"ID:\s*([a-f0-9]{16,})", line)
        if m:
            JOB_ID = m.group(1)
proc.wait()

print("=" * 72)
print(f"Job exit code: {proc.returncode}")
print(f"Job ID:        {JOB_ID}")
if JOB_ID:
    print(f"Job page:      https://huggingface.co/jobs/{USERNAME}/{JOB_ID}")

## 4. (Optional) Re-fetch Logs of a Past Job

If your Colab disconnected during streaming, paste the job ID below and re-run this cell to fetch the logs after the fact.

In [ ]:
# JOB_ID = "paste-job-id-here"  # uncomment + edit if Colab disconnected

if JOB_ID:
    res = subprocess.run(
        ["hf", "jobs", "logs", "--tail", "60", JOB_ID],
        capture_output=True, text=True,
    )
    print(res.stdout or res.stderr)
    print("\n--- inspect ---")
    res2 = subprocess.run(
        ["hf", "jobs", "inspect", JOB_ID],
        capture_output=True, text=True,
    )
    print(res2.stdout or res2.stderr)
else:
    print("No JOB_ID set — fill it in above and re-run.")

## 5. Download Results from HuggingFace Hub

The job uploads CSVs, JSON tables, and PNG plots to the model repo under `results/`.

In [ ]:
from huggingface_hub import snapshot_download

RESULTS_REPO = "yashvyasop/designgym2-grpo-qwen05-lora"

local_root = snapshot_download(
    repo_id=RESULTS_REPO,
    repo_type="model",
    allow_patterns=["results/*"],
    token=HF_TOKEN,
    local_dir="/content/grpo_results",
)
results_dir = os.path.join(local_root, "results")
print(f"Downloaded to: {results_dir}\n")
print("Files:")
for f in sorted(os.listdir(results_dir)):
    print(f"  {f}")

## 6. 3-Way Comparison Table

In [ ]:
import pandas as pd
from IPython.display import display

table = pd.read_csv(f"{results_dir}/comparison_summary_table.csv")
policy_order = ["base_qwen05", "sft_qwen05", "grpo_qwen05"]
table = table.set_index("policy").loc[policy_order].reset_index()

print("=" * 80)
print("            DesignGym: Base → SFT → GRPO   Comparison")
print("=" * 80)

display(table.style
    .format({
        "final_score": "{:.4f}",
        "instruction_score": "{:.4f}",
        "total_reward": "{:.2f}",
        "valid_json_rate": "{:.2%}",
        "valid_action_type_rate": "{:.2%}",
        "env_rejection_rate": "{:.2%}",
        "early_finalize_count": "{:.1f}",
        "invalid_actions": "{:.1f}",
    })
    .highlight_max(
        subset=["final_score", "instruction_score", "total_reward",
                "valid_json_rate", "valid_action_type_rate"],
        color="#c8e6c9")
    .highlight_min(
        subset=["env_rejection_rate", "early_finalize_count", "invalid_actions"],
        color="#c8e6c9")
    .set_caption("Higher is better for top metrics; lower is better for bottom metrics (green = best)"))

## 7. Per-Task Breakdown (3 tasks × 3 policies = 9 rows)

In [ ]:
by_task_path = f"{results_dir}/comparison_by_task.csv"
if os.path.exists(by_task_path):
    by_task = pd.read_csv(by_task_path)
    by_task["policy"] = pd.Categorical(by_task["policy"], categories=policy_order, ordered=True)
    by_task = by_task.sort_values(["policy", "task_id"]).reset_index(drop=True)
    display(by_task[["policy", "task_id", "final_score", "instruction_score",
                      "valid_json_rate", "total_reward"]].style
        .format({
            "final_score": "{:.4f}",
            "instruction_score": "{:.4f}",
            "valid_json_rate": "{:.2%}",
            "total_reward": "{:.2f}",
        })
        .background_gradient(subset=["final_score"], cmap="Greens")
        .set_caption("Per-task results"))
else:
    print("comparison_by_task.csv not present yet — re-run after job finishes.")

## 8. Comparison Plots (auto-generated by training script)

In [ ]:
from IPython.display import Image, display as idisplay
import glob

plot_files = sorted(glob.glob(f"{results_dir}/*.png"))
print(f"Found {len(plot_files)} plots\n")

for fp in plot_files:
    print(f"━━━ {os.path.basename(fp)} ━━━")
    idisplay(Image(filename=fp))

## 9. Summary & Improvements

In [ ]:
print("\n" + "=" * 60)
print("          FINAL RESULTS SUMMARY")
print("=" * 60)

for pol, label in zip(policy_order, ["BASE", "SFT ", "GRPO"]):
    row = table[table["policy"] == pol].iloc[0]
    print(f"\n  {label}")
    print(f"    final_score:           {row['final_score']:.4f}")
    print(f"    instruction_score:     {row['instruction_score']:.4f}")
    print(f"    valid_json_rate:       {row['valid_json_rate']:.2%}")
    print(f"    valid_action_type:     {row['valid_action_type_rate']:.2%}")
    print(f"    total_reward:          {row['total_reward']:.2f}")
    print(f"    env_rejection_rate:    {row['env_rejection_rate']:.2%}")

base_fs = float(table[table['policy']=='base_qwen05']['final_score'].iloc[0])
sft_fs  = float(table[table['policy']=='sft_qwen05']['final_score'].iloc[0])
grpo_fs = float(table[table['policy']=='grpo_qwen05']['final_score'].iloc[0])
base_is = float(table[table['policy']=='base_qwen05']['instruction_score'].iloc[0])
sft_is  = float(table[table['policy']=='sft_qwen05']['instruction_score'].iloc[0])
grpo_is = float(table[table['policy']=='grpo_qwen05']['instruction_score'].iloc[0])

def pct(a, b):
    return f"{(a-b)/b*100:+.1f}%" if b else "n/a"

print("\n  Final score deltas:")
print(f"    Δ Base→SFT:    {sft_fs-base_fs:+.4f}  ({pct(sft_fs, base_fs)})")
print(f"    Δ SFT→GRPO:    {grpo_fs-sft_fs:+.4f}  ({pct(grpo_fs, sft_fs)})")
print(f"    Δ Base→GRPO:   {grpo_fs-base_fs:+.4f}  ({pct(grpo_fs, base_fs)})")

print("\n  Instruction score deltas:")
print(f"    Δ Base→SFT:    {sft_is-base_is:+.4f}  ({pct(sft_is, base_is)})")
print(f"    Δ SFT→GRPO:    {grpo_is-sft_is:+.4f}  ({pct(grpo_is, sft_is)})")
print(f"    Δ Base→GRPO:   {grpo_is-base_is:+.4f}  ({pct(grpo_is, base_is)})")

print("\n" + "=" * 60)
print(f"  Trained adapter:  https://huggingface.co/{RESULTS_REPO}")
if JOB_ID:
    print(f"  Job logs:         https://huggingface.co/jobs/{USERNAME}/{JOB_ID}")
print("=" * 60)